# Intro a elsapy

## Bibliotecas

In [1]:
from elsapy.elsclient import ElsClient
from elsapy.elssearch import ElsSearch
from elsapy.elsprofile import ElsAuthor, ElsAffil
from elsapy.elsdoc import FullDoc, AbsDoc

import json

## Configurar Cliente
Utilizar la API key proporcioanda por Scopus, esta solo funciona al utilizar la red de la universidad, por lo cual para trabajar desde fuera de la U' se puede utilizar OpenVPN para conectarse a la red de la EPN.

**Revisar:** [Manual de Usuario para el acceso a OpenVPN - DGIP EPN](https://servicios-it.epn.edu.ec/images/DGIP/Descargas/MANUAL_de_VPN.pdf)


**Nota:** para este caso se está utilizando la API key proporcionada por Scopus en _Interactive APIs_, de [Elsevier Developer Portal](https://dev.elsevier.com/api_docs.html)

In [2]:
apikey = "7f59af901d2d86f78a1fd60c1bf9426a" # API key publica

## Inicializar cliente
client = ElsClient(apikey)
client.inst_token = ''

## Definiendo el flujo de obtención de datos
* [1] Buscar con ElsSearch (por nombre de autor, afiliación, artículo, etc.)
* [2] Extraer el ID del resultado (por ejemplo: AUTHOR_ID:123456789)
* [3] Usar ElsAuthor, ElsAffil, etc., con ese ID

```c#
    [A] Buscar afiliaciones de Ecuador --> [B] Buscar autores por afiliación --> [C] Recuperar Docs

Modelo de datos de la API de Scopus
```c#
         -------------------------------------------
        |                                           |
        V                                           V
    [Author]    <-->    [Affiliation]   <-->    [Article]

### 1. Obtener Afiliaciones

In [ ]:
# A
query = "AFFILCOUNTRY(Ecuador)"     # Filtro: pais de afiliacion
target = "affiliation"              # Tipo de búsqueda: por afiliación (API: Affiliation_Search)
search = ElsSearch(query, target)   # Ejecutar búsqueda con la clase ElsSearch (mecanismo basico de búsqueda)
search.execute(client)
print ("search has", search.num_res, "out of", search.tot_num_res, "potential results")
print("Has all results?", search.hasAllResults())

search has 25 out of 7974 potential results
Has all results? False


In [17]:
search.results

[{'@_fa': 'true',
  'link': [{'@_fa': 'true',
    '@ref': 'self',
    '@href': 'https://api.elsevier.com/content/affiliation/affiliation_id/60072059'},
   {'@_fa': 'true',
    '@ref': 'search',
    '@href': 'https://api.elsevier.com/content/search/scopus?query=af-id%2860072059%29'},
   {'@_fa': 'true',
    '@ref': 'scopus-affiliation',
    '@href': 'https://www.scopus.com/affil/profile.uri?afid=60072059&partnerID=HzOxMe3b&origin=inward'}],
  'prism:url': 'https://api.elsevier.com/content/affiliation/affiliation_id/60072059',
  'dc:identifier': 'AFFILIATION_ID:60072059',
  'eid': '10-s2.0-60072059',
  'affiliation-name': 'Universidad San Francisco de Quito',
  'name-variant': [{'@_fa': 'true', '$': 'San Francisco University of Quito'}],
  'document-count': '5963',
  'city': 'Quito',
  'country': 'Ecuador',
  'parent-affiliation-id': '0'},
 {'@_fa': 'true',
  'link': [{'@_fa': 'true',
    '@ref': 'self',
    '@href': 'https://api.elsevier.com/content/affiliation/affiliation_id/60072061'}

In [ ]:
afiliaciones = []
afiliaciones_ids = []

for result in search.results:
    print("Nombre:", result.get("affiliation-name"))
    afiliaciones.append(result.get("affiliation-name"))
    print("ID:", result.get("dc:identifier").split(":")[1])
    afiliaciones_ids.append(result.get("dc:identifier").split(":")[1])
    print("Ciudad:", result.get("city"))
    print("País:", result.get("country"))
    print("-" * 40)

Nombre: Universidad San Francisco de Quito
ID: 60072059
Ciudad: Quito
País: Ecuador
----------------------------------------
Nombre: Escuela Superior Politecnica del Litoral Ecuador
ID: 60072061
Ciudad: Guayaquil
País: Ecuador
----------------------------------------
Nombre: Escuela Politécnica Nacional
ID: 60072054
Ciudad: Quito
País: Ecuador
----------------------------------------
Nombre: Pontificia Universidad Católica del Ecuador
ID: 60072063
Ciudad: Quito
País: Ecuador
----------------------------------------
Nombre: Universidad de las Fuerzas Armadas ESPE
ID: 60104598
Ciudad: Sangolquí
País: Ecuador
----------------------------------------
Nombre: Universidad Técnica Particular de Loja
ID: 60072064
Ciudad: Loja
País: Ecuador
----------------------------------------
Nombre: Universidad de las Americas - Ecuador
ID: 60104441
Ciudad: Quito
País: Ecuador
----------------------------------------
Nombre: University of Cuenca
ID: 60072035
Ciudad: Cuenca
País: Ecuador
------------------

In [24]:
afiliaciones

['Universidad San Francisco de Quito',
 'Escuela Superior Politecnica del Litoral Ecuador',
 'Escuela Politécnica Nacional',
 'Pontificia Universidad Católica del Ecuador',
 'Universidad de las Fuerzas Armadas ESPE',
 'Universidad Técnica Particular de Loja',
 'Universidad de las Americas - Ecuador',
 'University of Cuenca',
 'Universidad Espíritu Santo',
 'Universidad Central del Ecuador',
 'Universidad Politécnica Salesiana, Cuenca',
 'Universidad de Guayaquil',
 'Universidad Técnica de Ambato',
 'Universidad Regional Autónoma de los Andes',
 'Universidad Técnica de Manabí',
 'Escuela Superior Politécnica de Chimborazo',
 'Yachay University for Experimental Technology and Research (Yachay Tech)',
 'Universidad Tecnológica Indoamérica',
 'Universidad UTE',
 'Universidad Católica de Cuenca',
 'Universidad Catolica de Santiago de Guayaquil',
 'Universidad Nacional de Chimborazo',
 'Universidad del Azuay',
 'Universidad Técnica del Norte',
 'Universidad Técnica Estatal de Quevedo']

### 2. Buscar Autores para cada Afiliación

In [ ]:
# B
afiliaciones_ids_test = ["60072054", "60072059"] # EPN, USFQ
results = []
for affil_id in afiliaciones_ids_test:
        print(f"Consultando autores de afiliación {affil_id}...")
        query = f"af-id({affil_id})"              # Filtro: por ID de afiliación
        print("Query:", query)
        target = "author"                       # Tipo de búsqueda: por autor (API: Author_Search)
        search = ElsSearch(query, target)
        search.execute(client)
        print("search has", search.num_res, "out of", search.tot_num_res, "potential results")
        results.extend(search.results)

Consultando autores de afiliación 60072054...
Query: af-id(60072054)
search has 25 out of 2020 potential results
Consultando autores de afiliación 60072059...
Query: af-id(60072059)
search has 25 out of 1923 potential results


In [ ]:
for result in results:
    print("ID:", result.get("dc:identifier").split(":")[1])
    print("Nombre:", result.get("preferred-name", {}).get("surname"), result.get("preferred-name", {}).get("given-name"))
    print("Afiliación:", result.get("affiliation-current", {}).get("affiliation-name"))
    # print("Areas de especialización:", result.get("subject-area", []))
    # Subject areas ineficientes, es mejor considerar usar Author_Retrieval
    subject_areas = result.get("subject-area", [])
    nombres_areas = list({area.get("$") for area in subject_areas if "$" in area})
    print("Áreas de especialización:", ", ".join(nombres_areas))
    print("-" * 40)

ID: 8154044400
Nombre: Ayala Edy
Afiliación: Escuela Politécnica Nacional
Áreas de especialización: Physics and Astronomy (all)
----------------------------------------
ID: 6603155100
Nombre: Mothes Patricia A.
Afiliación: Escuela Politécnica Nacional
Áreas de especialización: Earth and Planetary Sciences (all)
----------------------------------------
ID: 6701720423
Nombre: Ruales Jenny
Afiliación: Escuela Politécnica Nacional
Áreas de especialización: Agricultural and Biological Sciences (all), Medicine (all)
----------------------------------------
ID: 24782346900
Nombre: Hahn Friedrich L.
Afiliación: Escuela Politécnica Nacional
Áreas de especialización: Biochemistry, Genetics and Molecular Biology (all), Chemistry (all)
----------------------------------------
ID: 36187649600
Nombre: Yoo Sang Guun
Afiliación: Escuela Politécnica Nacional
Áreas de especialización: Computer Science (all), Engineering (all)
----------------------------------------
ID: 14019356600
Nombre: Donoso David 

### 2.1 Obtener detalles de autores

Para tener datos más detallados de un autor, ya que en el caso anterior no es muy util la información de áreas de especialización, es por ello
que se puede usar la API Author_Retrieval para mejorar los datos de los autores.

ID: 8154044400  
Nombre: Ayala Edy  
Afiliación: Escuela Politécnica Nacional  
Áreas de especialización: Physics and Astronomy (all)

In [ ]:
auth_id = "8154044400"  # ID del autor a consultar
auth_retrieval = ElsAuthor(author_id=auth_id)

In [17]:
if auth_retrieval.read(client):
    subject_areas = auth_retrieval.data.get('subject-areas', {}).get('subject-area', [])
    for sa in subject_areas:
        print(f"Área: {sa.get('@abbrev')}, Nombre: {sa.get('$')}, Código: {sa.get('@code')}")
else:
    print("Read author failed.")

Área: PHYS, Nombre: Nuclear and High Energy Physics, Código: 3106
Área: CHEM, Nombre: Spectroscopy, Código: 1607
Área: MULT, Nombre: Multidisciplinary, Código: 1000
Área: PHYS, Nombre: Instrumentation, Código: 3105
Área: MATH, Nombre: Mathematical Physics, Código: 2610
Área: ENGI, Nombre: Engineering (miscellaneous), Código: 2201
Área: COMP, Nombre: Software, Código: 1712
Área: CHEM, Nombre: Analytical Chemistry, Código: 1602
Área: COMP, Nombre: Computer Science (miscellaneous), Código: 1701
Área: MEDI, Nombre: Medicine (all), Código: 2700
Área: PHYS, Nombre: Physics and Astronomy (miscellaneous), Código: 3101
Área: PHYS, Nombre: Physics and Astronomy (all), Código: 3100


### 3. Obtener data de docs Scopus

In [ ]:
# C
query = "AU-ID(8154044400)"         # Filtro: Id del autor
target = "scopus"                   # Tipo de búsqueda: por documentos (API: Scopus_Search)
search = ElsSearch(query, target)
search.execute(client)
print ("search has", search.num_res, "out of", search.tot_num_res, "potential results")
print("Has all results?", search.hasAllResults())

search has 25 out of 553 potential results
Has all results? False


In [8]:
for doc in search.results:
    print("ID:", doc.get("dc:identifier").split(":")[1])
    print("Título:", doc.get("dc:title"))
    print("Fecha de publicación:", doc.get("prism:coverDate"))
    print("Fuente:", doc.get("prism:publicationName"))
    print("DOI:", doc.get("prism:doi"))
    print("-" * 40)

ID: 105009650623
Título: Multiplicity dependence of charm baryon and charm meson production in pPb collisions at sNN=8.16TeV
Fecha de publicación: 2025-09-01
Fuente: Physics Letters Section B Nuclear Elementary Particle and High Energy Physics
DOI: 10.1016/j.physletb.2025.139672
----------------------------------------
ID: 105007653404
Título: Determination of the strong coupling and its running from measurements of inclusive jet production
Fecha de publicación: 2025-09-01
Fuente: Physics Letters Section B Nuclear Elementary Particle and High Energy Physics
DOI: 10.1016/j.physletb.2025.139651
----------------------------------------
ID: 105005631044
Título: Observation of nuclear modification of energy-energy correlators inside jets in heavy ion collisions
Fecha de publicación: 2025-07-01
Fuente: Physics Letters Section B Nuclear Elementary Particle and High Energy Physics
DOI: 10.1016/j.physletb.2025.139556
----------------------------------------
ID: 105005652270
Título: Search for h